In [1]:
import warnings
warnings.filterwarnings("ignore")
 
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch
from scipy.optimize import minimize
from scipy.stats import pearsonr, spearmanr
from sklearn.decomposition import PCA
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel, WhiteKernel
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

In [2]:
#Variables
N_POINTS   = 100          # chord-wise stations per surface
N_PGA      = 12           # PGA (PCA) components to retain
SEED       = 42
 
# NREL 5MW operating envelope
# Rated power   : 5 MW
# Rotor diameter: 126 m
# Rated wind    : 11.4 m/s   Cut-in: 3 m/s   Cut-out: 25 m/s
# Dominant blade sections: r/R = 0.4–0.85  (main load-bearing region)
# Section chord-based Re  : ~3–6 × 10^6
# Operating AoA range     : 4°–10° (below stall, above zero-lift)
NREL_AoA_MIN   =  4.0    # deg — cut-in regime
NREL_AoA_NOM   =  7.0    # deg — rated-wind nominal AoA
NREL_AoA_MAX   = 12.0    # deg — gusts / partial-load operation
NREL_ALPHA_DEG = np.array([4, 5, 6, 7, 8, 9, 10, 11, 12])   # operating sweep
NREL_WEIGHTS   = np.array([0.04, 0.08, 0.14, 0.20, 0.20,    # Rayleigh-weighted
                            0.15, 0.10, 0.06, 0.03])          # probability
NREL_WEIGHTS  /= NREL_WEIGHTS.sum()                           # normalise
 
rng = np.random.default_rng(SEED)
 
# colours
C1, C2, C3, C4, C5 = "#2196F3", "#F44336", "#4CAF50", "#FF9800", "#9C27B0"
DARK, PANEL, BRD    = "#0d1117", "#161b22", "#30363d"

In [3]:
#Load Data and Geometry Extracting

shapes  = np.load("curated_airfoils.npz")["shapes"]   # (19164, 1001, 2)
classes = np.load("curated_airfoils.npz")["classes"]  # (19164,)
 
x_grid = (1 - np.cos(np.linspace(0, np.pi, N_POINTS))) / 2   # cosine spacing

def extract_geometry(shape: np.ndarray):
    """
    Returns a dict of scalar geometric parameters + interpolated surfaces.
    Layout: indices 0..LE  = lower surface (TE->LE),  LE..end = upper (LE->TE).
    """
    x, y   = shape[:, 0], shape[:, 1]
    le_idx = int(np.argmin(x))
 
    x_lo = x[:le_idx + 1][::-1].copy()
    y_lo = y[:le_idx + 1][::-1].copy()
    x_up = x[le_idx:].copy()
    y_up = y[le_idx:].copy()
 
    def dedup(xa, ya):
        _, k = np.unique(xa, return_index=True)
        return xa[k], ya[k]
 
    x_up, y_up = dedup(x_up, y_up)
    x_lo, y_lo = dedup(x_lo, y_lo)
 
    if len(x_up) < 3 or len(x_lo) < 3:
        return None
    if not (np.all(np.diff(x_up) > 0) and np.all(np.diff(x_lo) > 0)):
        return None
 
    try:
        yu = np.interp(x_grid, x_up, y_up)
        yl = np.interp(x_grid, x_lo, y_lo)
    except Exception:
        return None
 
    t  = yu - yl                          # thickness distribution
    c  = (yu + yl) / 2.0                  # camber line
 
    if t.max() < 0.005 or t.max() > 0.60:
        return None
 
    te_thick   = float(abs(yu[-1] - yl[-1]))
    max_t      = float(t.max())
    x_max_t    = float(x_grid[np.argmax(t)])
    max_camber = float(abs(c).max())
    le_radius  = float(t[2])              # thickness at 1st interior station
    te_slope   = float((yu[-5] - yl[-5]) - (yu[-1] - yl[-1]))   # taper rate
 
    return dict(
        yu=yu, yl=yl, thickness=t, camber=c,
        te_thick=te_thick, max_t=max_t, x_max_t=x_max_t,
        max_camber=max_camber, le_radius=le_radius, te_slope=te_slope,
    )

In [4]:
# Build Y_vecs and scalar arrays in one pass, then free shapes immediately
_te_thick, _max_t, _x_max_t = [], [], []
_max_camber, _le_radius, _te_slope = [], [], []
_Y_vecs_rows = []
valid_idx = []
 
for i in range(len(shapes)):
    shape = shapes[i]
    x, y  = shape[:, 0], shape[:, 1]
    le    = int(np.argmin(x))
    x_lo  = x[:le+1][::-1].copy(); y_lo = y[:le+1][::-1].copy()
    x_up  = x[le:].copy();         y_up = y[le:].copy()
 
    def _dedup(xa, ya):
        _, k = np.unique(xa, return_index=True); return xa[k], ya[k]
    x_up, y_up = _dedup(x_up, y_up)
    x_lo, y_lo = _dedup(x_lo, y_lo)
 
    if len(x_up) < 3 or len(x_lo) < 3:
        continue
    if not (np.all(np.diff(x_up) > 0) and np.all(np.diff(x_lo) > 0)):
        continue
    try:
        yu = np.interp(x_grid, x_up, y_up)
        yl = np.interp(x_grid, x_lo, y_lo)
    except Exception:
        continue
 
    t = yu - yl; c = (yu + yl) / 2
    if t.max() < 0.005 or t.max() > 0.60:
        continue
 
    _te_thick.append(float(abs(yu[-1] - yl[-1])))
    _max_t.append(float(t.max()))
    _x_max_t.append(float(x_grid[np.argmax(t)]))
    _max_camber.append(float(abs(c).max()))
    _le_radius.append(float(t[2]))
    _te_slope.append(float((yu[-5] - yl[-5]) - (yu[-1] - yl[-1])))
    _Y_vecs_rows.append(np.concatenate([yu, yl]).astype(np.float32))
    valid_idx.append(i)
del shapes   # free 307 MB immediately after extraction

In [5]:
valid_idx  = np.array(valid_idx)
valid_cls  = classes[valid_idx]; del classes
N_valid    = len(valid_idx)
 
# Scalar geometry arrays
te_thick   = np.array(_te_thick);   del _te_thick
max_t      = np.array(_max_t);      del _max_t
x_max_t    = np.array(_x_max_t);    del _x_max_t
max_camber = np.array(_max_camber); del _max_camber
le_radius  = np.array(_le_radius);  del _le_radius
te_slope   = np.array(_te_slope);   del _te_slope
 
# Surface y-vector matrix — used only for PCA
Y_vecs = np.array(_Y_vecs_rows, dtype=np.float32); del _Y_vecs_rows
 
# We also need geom_list for visualisation — rebuild a lightweight version
# storing only what's needed for figure plotting
geom_list = [{"yu": Y_vecs[i, :N_POINTS], "yl": Y_vecs[i, N_POINTS:]}
             for i in range(N_valid)]

In [6]:
print(f"  Valid airfoils : {N_valid:,}")
 
print(f"\n  TE thickness  — min:{te_thick.min()*100:.3f}%  "
      f"median:{np.median(te_thick)*100:.3f}%  max:{te_thick.max()*100:.3f}%")
print(f"  Max thickness — min:{max_t.min()*100:.1f}%  "
      f"median:{np.median(max_t)*100:.1f}%  max:{max_t.max()*100:.1f}%")

  Valid airfoils : 19,164

  TE thickness  — min:0.000%  median:0.426%  max:6.223%
  Max thickness — min:2.0%  median:23.8%  max:55.4%


PHYSICS-BASED Cl / Cd OVER THE NREL 5MW OPERATING ENVELOPE

In [7]:
alpha_full = np.linspace(-10, 25, 36)   # full polar for visualisation
alpha_op   = NREL_ALPHA_DEG             # operating envelope
 
def kirchhoff_cl(alpha_d, stall_d, Cl_max, Cl_alpha, al0):
    """Kirchhoff stall model (vectorised over alpha)."""
    ar    = np.deg2rad(alpha_d)
    asr   = np.deg2rad(stall_d)
    Cl_lin = Cl_alpha * (ar - al0)
    delta  = ar - asr
    f      = np.where(delta <= 0, 1.0,
                      np.exp(-2.5 * delta / (0.1 + 0.3 * asr)))
    f      = np.clip(f, 0, 1)
    Cl_k   = Cl_max * ((1 + np.sqrt(f)) / 2) ** 2
    blend  = np.where(delta <= 0, 1.0,
                      np.exp(-4.0 * np.clip(delta, 0, None)))
    Cl     = blend * Cl_lin + (1 - blend) * Cl_k
    f_neg  = np.exp(-2.5 * np.abs(np.minimum(delta, 0)) / (0.1 + 0.3 * asr))
    return np.where(alpha_d < -stall_d,
                    -Cl_max * ((1 + np.sqrt(np.clip(f_neg, 0, 1))) / 2) ** 2,
                    Cl)
 
def compute_cd(alpha_d, stall_d, Cd0, K, Cl_arr):
    """Empirical drag polar + post-stall surge."""
    delta    = alpha_d - stall_d
    Cd_polar = Cd0 + K * Cl_arr ** 2
    surge    = (0.8 * (1 - np.exp(-np.maximum(0, delta) / 3))
                * np.sin(np.deg2rad(np.abs(alpha_d))) ** 2)
    return np.clip(Cd_polar + surge, 0.003, 3.0)

In [8]:
# Per-airfoil aerodynamic parameters
alpha_L0_arr = -(2.0 * max_camber + 0.5 * max_camber * 0.5)
Cl_alpha_arr = 2.0 * np.pi * (1.0 + 0.77 * max_t)
alpha_stall  = np.clip(8 + 20 * le_radius + 5 * max_t - 3 * max_camber, 8, 18)
Cl_max_arr   = 1.0 + 2.0 * max_camber + 0.3 * max_t
 
# Trailing-edge effect on drag:
#   Blunt TE creates a turbulent wake → higher Cd0
#   Effect magnitude calibrated from XFoil experiments:
#     ΔCd0 ≈ 0.5 * (te/c)^2  (quadratic penalty)
Cd0_base = np.clip(0.004 + 0.008 * max_t + 0.01 * max_t ** 2, 0.003, 0.05)
Cd0_te   = 0.5 * te_thick ** 2          # TE bluntness penalty
Cd0_arr  = Cd0_base + Cd0_te
K_arr    = 0.012 + 0.03 * max_camber + 0.01 * np.abs(te_slope)
 
# Full polars — (N_valid, 36)
Cl_full = np.zeros((N_valid, len(alpha_full)))
Cd_full = np.zeros((N_valid, len(alpha_full)))
for i in range(N_valid):
    Cl_full[i] = kirchhoff_cl(alpha_full, alpha_stall[i],
                               Cl_max_arr[i], Cl_alpha_arr[i], alpha_L0_arr[i])
    Cd_full[i] = compute_cd(alpha_full, alpha_stall[i],
                             Cd0_arr[i], K_arr[i], Cl_full[i])
# Operating-envelope polars — (N_valid, 9)
Cl_op = np.zeros((N_valid, len(alpha_op)))
Cd_op = np.zeros((N_valid, len(alpha_op)))
for i in range(N_valid):
    Cl_op[i] = kirchhoff_cl(alpha_op, alpha_stall[i],
                              Cl_max_arr[i], Cl_alpha_arr[i], alpha_L0_arr[i])
    Cd_op[i] = compute_cd(alpha_op, alpha_stall[i],
                           Cd0_arr[i], K_arr[i], Cl_op[i])
    
LD_full = Cl_full / np.clip(Cd_full, 1e-4, None)
LD_op   = Cl_op   / np.clip(Cd_op,   1e-4, None)
 
# Rayleigh-weighted mean L/D over the operating envelope
LD_weighted = (LD_op * NREL_WEIGHTS[None, :]).sum(axis=1)
 
print(f"  Cl range (op): [{Cl_op.min():.3f}, {Cl_op.max():.3f}]")
print(f"  Cd range (op): [{Cd_op.min():.5f}, {Cd_op.max():.4f}]")
print(f"  Weighted L/D : [{LD_weighted.min():.1f}, {LD_weighted.max():.1f}]")

  Cl range (op): [0.446, 3.313]
  Cd range (op): [0.00656, 0.1896]
  Weighted L/D : [18.5, 58.2]


TRAILING-EDGE THICKNESS IMPACT

In [9]:
# ── Bin airfoils by TE thickness ──────────────────────────────────────────
te_percentiles = [0, 10, 25, 50, 75, 90, 100]
te_bins_edges  = np.percentile(te_thick, te_percentiles)
 
bin_labels = [
    "Sharp (0–10%)",
    "Low   (10–25%)",
    "Mod   (25–50%)",
    "High  (50–75%)",
    "Blunt (75–90%)",
    "Very blunt (90–100%)",
]
 
te_bin_idx = np.digitize(te_thick, te_bins_edges[1:-1])   # 0-based bin index
 
print("\n  TE Thickness Bins:")
print(f"  {'Bin':<22} {'Count':>6}  {'TE mean':>9}  {'wt-L/D':>8}  {'Cd0 mean':>9}")
print("  " + "-" * 60)
bin_stats = {}
for b, lbl in enumerate(bin_labels):
    mask = te_bin_idx == b
    if mask.sum() == 0:
        continue
    te_m  = te_thick[mask].mean() * 100
    ld_m  = LD_weighted[mask].mean()
    cd0_m = Cd0_arr[mask].mean()
    bin_stats[b] = dict(mask=mask, label=lbl, te_mean=te_m,
                        ld_mean=ld_m, cd0_mean=cd0_m, n=mask.sum())
    print(f"  {lbl:<22} {mask.sum():>6}  {te_m:>8.3f}%  {ld_m:>8.2f}  {cd0_m:>9.5f}")


  TE Thickness Bins:
  Bin                     Count    TE mean    wt-L/D   Cd0 mean
  ------------------------------------------------------------
  Sharp (0–10%)            1917     0.000%     44.88    0.00547
  Low   (10–25%)           2874     0.000%     47.20    0.00516
  Mod   (25–50%)           4187     0.150%     43.98    0.00563
  High  (50–75%)           5166     0.548%     41.77    0.00742
  Blunt (75–90%)           3012     1.363%     42.21    0.00756
  Very blunt (90–100%)     2008     2.218%     39.10    0.00936


In [10]:
r_pearson, p_pearson   = pearsonr(te_thick, LD_weighted)
r_spearman, p_spearman = spearmanr(te_thick, LD_weighted)
 
print(f"\n Pearson  r = {r_pearson:.4f}  (p={p_pearson:.2e})")
print(f"  Spearman r = {r_spearman:.4f}  (p={p_spearman:.2e})")


 Pearson  r = -0.4162  (p=0.00e+00)
  Spearman r = -0.5148  (p=0.00e+00)


In [11]:
# Partial correlation controlling for max thickness
from numpy.linalg import lstsq
 
def partial_corr(y, x, controls):
    """Partial Pearson r of y~x after removing linear effect of controls."""
    Z = np.column_stack(controls)
    y_res = y - Z @ lstsq(Z, y, rcond=None)[0]
    x_res = x - Z @ lstsq(Z, x, rcond=None)[0]
    return pearsonr(x_res, y_res)
 
r_partial, p_partial = partial_corr(LD_weighted, te_thick, [max_t, max_camber, x_max_t])
print(f"  Partial r  = {r_partial:.4f}  (p={p_partial:.2e})"
      f"  [controlling for max_t, camber, x_max_t]")

  Partial r  = 0.0218  (p=2.49e-03)  [controlling for max_t, camber, x_max_t]


In [13]:
# ── Sensitivity analysis: isolate TE effect ───────────────────────────────
# Among airfoils with similar max_t (18–22%), compute LD vs TE thickness
mask_iso = (max_t >= 0.18) & (max_t <= 0.22) & (max_camber <= 0.04)
te_iso   = te_thick[mask_iso]
ld_iso   = LD_weighted[mask_iso]
 
lr_iso   = LinearRegression().fit(te_iso.reshape(-1, 1), ld_iso)
slope_iso = lr_iso.coef_[0]
r2_iso    = r2_score(ld_iso, lr_iso.predict(te_iso.reshape(-1, 1)))
 
print(f"\n  Isolated effect (18–22% thick, camber ≤ 4%, n={mask_iso.sum()}):")
print(f"    ΔL/D per 1% TE = {slope_iso:.2f}  (R^2={r2_iso:.3f})")
print(f"    Interpretation: increasing TE by 1%c changes L/D by {slope_iso:.2f}")


  Isolated effect (18–22% thick, camber ≤ 4%, n=2768):
    ΔL/D per 1% TE = -402.03  (R^2=0.078)
    Interpretation: increasing TE by 1%c changes L/D by -402.03
